In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, average_precision_score, PrecisionRecallDisplay, precision_recall_curve

## Load data

In [3]:
df2 = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv") #Local
df2.head(2)

,child_age,child_weight,diarrhoea_last_2_weeks,fever_last_2_weeks,area,child_sex,mother_education,health_insurance,wealth_index,malnurished,province_1.0,province_2.0,province_3.0,province_4.0,province_5.0,province_6.0,province_7.0
0,1,-1.085628,0,0,0,1,5,0,1,1,1,0,0,0,0,0,0
1,3,0.420314,0,1,0,1,5,0,1,1,1,0,0,0,0,0,0


### Train-test Split

In [4]:
X = df2.drop(columns=['malnurished'])
y = df2['malnurished']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state=42)

# LightGBM

### Simple mode

In [6]:
lgbm = lgb.LGBMClassifier(random_state=42)
lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 2316, number of negative: 2828
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000271 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 207
[LightGBM] [Info] Number of data points in the train set: 5144, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.450233 -> initscore=-0.199728
[LightGBM] [Info] Start training from score -0.199728


LGBMClassifier(random_state=42)

In [7]:
y_pred = lgbm.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.81      0.78       707
           1       0.74      0.68      0.71       579

    accuracy                           0.75      1286
   macro avg       0.75      0.74      0.74      1286
weighted avg       0.75      0.75      0.75      1286



### Average Precision

In [8]:
y_probas = lgbm.predict_proba(X_test)[:, 1]
print(f'Average Precision: {average_precision_score(y_test, y_probas)}')

Average Precision: 0.8196469068218322


## Hyperparameter tuning

In [ ]:
# Define parameter grid to search
params = {
    'num_leaves': [15, 31, 63],
    'max_depth': [-1, 5, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 300, 500],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

lgb_params = LGBMClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=lgb_params,
    param_grid=params,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 2916 candidates, totalling 14580 fits


In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

In [ ]:
lgbm_tune = grid_search.best_estimator_
y_pred = lgbm_tune.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

## Average precision

In [ ]:
y_probas_tune = lgbm_tune.predict_proba(X_test)[:, 1]
print(f'Average Precision: {average_precision_score(y_test, y_probas_tune)}')

In [ ]:
y_probas_tune = lgbm_tune.predict_proba(X_test)[:, 1]
print(f'Average Precision: {average_precision_score(y_test, y_probas_tune)}')

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Nurished', 'Malnurished'], yticklabels=['Nurished', 'Malnurished'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgbm_tune.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)
print(feature_importance)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x='Importance', y='Feature', palette='viridis')
plt.title("Top 15 Feature Importances (LightGBM)")
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (by split or gain)
import lightgbm as lgb

lgb.plot_importance(lgbm_tune, max_num_features=15, importance_type='split', height=0.5)
plt.title("Feature Importance (by split)")
plt.show()